In [1]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Runtime setup successful.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas version: 2.2.3
NumPy version: 2.1.3
Runtime setup successful.


In [2]:
from google.colab import files

uploaded = files.upload()

Saving hrim_historical_training_baseline.csv to hrim_historical_training_baseline.csv


In [3]:
import pandas as pd
from pathlib import Path

data_path = Path("/content/hrim_historical_training_baseline.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["structural_change_proxy"].value_counts())

Dataset shape: (3266, 11)

Columns:
['repository', 'previous_snapshot', 'current_snapshot', 'package_path', 'previous_java_files', 'previous_nonempty_lines', 'previous_fan_in', 'previous_fan_out', 'previous_total_coupling', 'previous_instability', 'structural_change_proxy']

First 5 rows:


,repository,previous_snapshot,current_snapshot,package_path,previous_java_files,previous_nonempty_lines,previous_fan_in,previous_fan_out,previous_total_coupling,previous_instability,structural_change_proxy
0,R03_jabref,2020-01-08,2024-09-16,org.jabref,7,723,56,30,86,0.348837,1
1,R03_jabref,2020-01-08,2024-09-16,org.jabref.architecture,1,9,2,0,2,0.000000,1
2,R03_jabref,2020-01-08,2024-09-16,org.jabref.benchmarks,1,119,0,15,15,1.000000,0
3,R03_jabref,2020-01-08,2024-09-16,org.jabref.cli,6,887,2,28,30,0.933333,0
4,R03_jabref,2020-01-08,2024-09-16,org.jabref.gui,25,4120,47,77,124,0.620968,0



Missing values:
repository                 0
previous_snapshot          0
current_snapshot           0
package_path               0
previous_java_files        0
previous_nonempty_lines    0
previous_fan_in            0
previous_fan_out           0
previous_total_coupling    0
previous_instability       0
structural_change_proxy    0
dtype: int64

Target distribution:
structural_change_proxy
0    3030
1     236
Name: count, dtype: int64


Prepare baseline model

In [4]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

print("Scikit-learn and XGBoost imports successful.")

Scikit-learn and XGBoost imports successful.


Train and evaluate the historical baseline

In [5]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/hrim_historical_training_baseline.csv")

# Six historical architecture features
feature_cols = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

X = df[feature_cols]
y = df["structural_change_proxy"].astype(int)
groups = df["repository"]

logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "repository": groups.iloc[test_idx].iloc[0],
        "test_examples": len(test_idx),
        "positive_examples": int(y_test.sum()),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "pr_auc": average_precision_score(y_test, probabilities),
        "roc_auc": (
            roc_auc_score(y_test, probabilities)
            if y_test.nunique() == 2 else np.nan
        ),
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

print("\nMean metrics across held-out repositories:")
display(
    results_df[
        ["precision", "recall", "f1", "pr_auc", "roc_auc"]
    ].mean().to_frame("mean").round(4)
)

,repository,test_examples,positive_examples,precision,recall,f1,pr_auc,roc_auc
0,R03_jabref,329,37,0.2407,0.3514,0.2857,0.2330,0.6982
1,R04_argouml,143,24,0.1892,0.2917,0.2295,0.1805,0.4981
2,R05_lucene,401,33,0.1132,0.3636,0.1727,0.1179,0.5396
3,R06_hadoop,1006,43,0.1212,0.3721,0.1829,0.1164,0.7404
4,R07_junit,106,16,0.2571,0.5625,0.3529,0.3362,0.6448
5,R08_apache_maven,84,7,0.1622,0.8571,0.2727,0.2771,0.6883
6,R09_kafca,327,39,0.2209,0.4872,0.3040,0.2001,0.6594
7,R11_key_cloak,870,37,0.0938,0.4865,0.1572,0.0814,0.6877



Mean metrics across held-out repositories:


,mean
precision,0.1748
recall,0.4715
f1,0.2447
pr_auc,0.1928
roc_auc,0.6446


In [6]:
from google.colab import files

uploaded = files.upload()

Saving hrim_fused_features.csv to hrim_fused_features.csv


Inspect fuse features

In [9]:
from google.colab import files

uploaded = files.upload()

Saving historical.zip to historical.zip


Extract the historical ZIP file

In [10]:
import zipfile
from pathlib import Path

zip_path = Path("/content/historical.zip")
extract_path = Path("/content/data/historical")

if not zip_path.exists():
    print("ZIP file not found. Please check the uploaded filename.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    csv_files = list(extract_path.rglob("all_snapshots_architecture.csv"))

    print("Extraction completed.")
    print("Historical snapshot CSV files found:", len(csv_files))

    for file in csv_files:
        print(file)

Extraction completed.
Historical snapshot CSV files found: 11
/content/data/historical/historical/R09_kafca/all_snapshots_architecture.csv
/content/data/historical/historical/R03_jabref/all_snapshots_architecture.csv
/content/data/historical/historical/R02_common_lang/all_snapshots_architecture.csv
/content/data/historical/historical/R05_lucene/all_snapshots_architecture.csv
/content/data/historical/historical/R06_hadoop/all_snapshots_architecture.csv
/content/data/historical/historical/R11_key_cloak/all_snapshots_architecture.csv
/content/data/historical/historical/R04_argouml/all_snapshots_architecture.csv
/content/data/historical/historical/R01_spring_petclinic/all_snapshots_architecture.csv
/content/data/historical/historical/R08_apache_maven/all_snapshots_architecture.csv
/content/data/historical/historical/R10_teammates/all_snapshots_architecture.csv
/content/data/historical/historical/R07_junit/all_snapshots_architecture.csv


Inspect the historical snapshot coverage

In [11]:
from pathlib import Path
import pandas as pd

historical_root = Path("/content/data/historical/historical")

csv_files = sorted(
    historical_root.glob("*/all_snapshots_architecture.csv")
)

print("Repositories found:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file)

    print(
        f"{file.parent.name}: "
        f"{len(df)} rows, "
        f"{df['snapshot_date'].nunique()} snapshots, "
        f"{df['package_path'].nunique()} unique packages"
    )

    if file == csv_files[0]:
        print("\nColumn names:")
        print(df.columns.tolist())
        print("\nFirst five rows:")
        display(df.head())

Repositories found: 11
R01_spring_petclinic: 23 rows, 4 snapshots, 14 unique packages

Column names:
['repository', 'snapshot_date', 'commit_hash', 'package_path', 'java_files', 'nonempty_lines', 'fan_in', 'fan_out', 'total_coupling', 'instability']

First five rows:


,repository,snapshot_date,commit_hash,package_path,java_files,nonempty_lines,fan_in,fan_out,total_coupling,instability
0,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic,1,10,0,0,0,0.0
1,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.owner,4,80,0,1,1,1.0
2,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.pet,4,55,0,1,1,1.0
3,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.util,3,15,2,0,2,0.0
4,R01_spring_petclinic,2015-01-21,cc4ae966c63610344313e6c7bf81ae7219ab8370,org.springframework.samples.petclinic.model,11,621,7,0,7,0.0


R02_common_lang: 51 rows, 4 snapshots, 20 unique packages
R03_jabref: 792 rows, 4 snapshots, 462 unique packages
R04_argouml: 286 rows, 4 snapshots, 143 unique packages
R05_lucene: 921 rows, 4 snapshots, 512 unique packages
R06_hadoop: 1891 rows, 4 snapshots, 885 unique packages
R07_junit: 195 rows, 4 snapshots, 89 unique packages
R08_apache_maven: 405 rows, 4 snapshots, 280 unique packages
R09_kafca: 622 rows, 4 snapshots, 294 unique packages
R10_teammates: 80 rows, 4 snapshots, 58 unique packages
R11_key_cloak: 1770 rows, 4 snapshots, 895 unique packages


Build historical HRIM features

In [12]:
from pathlib import Path
import pandas as pd

historical_root = Path("/content/data/historical/historical")
csv_files = sorted(
    historical_root.glob("*/all_snapshots_architecture.csv")
)

frames = [pd.read_csv(file) for file in csv_files]
history = pd.concat(frames, ignore_index=True)

history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

print("Combined historical dataset:", history.shape)
print("Repositories:", history["repository"].nunique())
print("Snapshots:", history.groupby("repository")["snapshot_date"].nunique().sum())
print("Duplicate repository/date/package rows:",
      history.duplicated(
          ["repository", "snapshot_date", "package_path"]
      ).sum())
print("Missing values in required columns:",
      history[
          ["repository", "snapshot_date", "package_path",
           "java_files", "nonempty_lines", "fan_in",
           "fan_out", "total_coupling", "instability"]
      ].isna().sum().sum())

display(
    history.groupby("repository").agg(
        snapshots=("snapshot_date", "nunique"),
        package_snapshot_rows=("package_path", "size"),
        unique_packages=("package_path", "nunique")
    ).reset_index()
)

Combined historical dataset: (7036, 10)
Repositories: 11
Snapshots: 44
Duplicate repository/date/package rows: 0
Missing values in required columns: 0


,repository,snapshots,package_snapshot_rows,unique_packages
0,R01_spring_petclinic,4,23,14
1,R02_common_lang,4,51,20
2,R03_jabref,4,792,462
3,R04_argouml,4,286,143
4,R05_lucene,4,921,512
5,R06_hadoop,4,1891,885
6,R07_junit,4,195,89
7,R08_apache_maven,4,405,280
8,R09_kafca,4,622,294
9,R10_teammates,4,80,58


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

# Load all historical repository snapshots
historical_root = Path("/content/data/historical/historical")

frames = [
    pd.read_csv(file)
    for file in sorted(
        historical_root.glob("*/all_snapshots_architecture.csv")
    )
]

history = pd.concat(frames, ignore_index=True)
history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

metrics = [
    "java_files",
    "nonempty_lines",
    "fan_in",
    "fan_out",
    "total_coupling",
    "instability",
]

# Sort snapshots and package records chronologically
history = history.sort_values(
    ["repository", "package_path", "snapshot_date"]
).reset_index(drop=True)

# Find the previous observation of each package
grouped = history.groupby(["repository", "package_path"], sort=False)

for col in metrics:
    history[f"previous_{col}"] = grouped[col].shift(1)

history["previous_snapshot"] = grouped["snapshot_date"].shift(1)
history["previous_commit_hash"] = grouped["commit_hash"].shift(1)

# Keep only package observations present in consecutive snapshots.
# A gap means this is not necessarily an adjacent-snapshot transition.
history["previous_snapshot_count"] = grouped.cumcount()
history["current_snapshot_index"] = (
    history.groupby(["repository", "snapshot_date"])["snapshot_date"]
    .transform(lambda x: 0)
)

# Check which repositories have packages in every snapshot
coverage = (
    history.groupby(["repository", "snapshot_date"])["package_path"]
    .count()
    .reset_index(name="package_rows")
)

print("Snapshot package counts:")
display(coverage)

print(
    "\nRows with a previous observation:",
    int(history["previous_snapshot"].notna().sum())
)

Snapshot package counts:


,repository,snapshot_date,package_rows
0,R01_spring_petclinic,2013-01-09,4
1,R01_spring_petclinic,2015-01-21,8
2,R01_spring_petclinic,2021-05-12,6
3,R01_spring_petclinic,2026-09-29,5
4,R02_common_lang,2002-07-19,2
5,R02_common_lang,2011-08-16,13
6,R02_common_lang,2022-05-04,18
7,R02_common_lang,2026-10-07,18
8,R03_jabref,2013-03-24,48
9,R03_jabref,2020-01-08,142



Rows with a previous observation: 3384


Build  aligned historical transitions

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

historical_root = Path("/content/data/historical/historical")

frames = [
    pd.read_csv(file)
    for file in sorted(
        historical_root.glob("*/all_snapshots_architecture.csv")
    )
]

history = pd.concat(frames, ignore_index=True)
history["snapshot_date"] = pd.to_datetime(history["snapshot_date"])

metrics = [
    "java_files",
    "nonempty_lines",
    "fan_in",
    "fan_out",
    "total_coupling",
    "instability",
]

transition_frames = []

for repository, repo_df in history.groupby("repository"):
    dates = sorted(repo_df["snapshot_date"].unique())

    for i in range(1, len(dates)):
        previous_date = pd.Timestamp(dates[i - 1])
        current_date = pd.Timestamp(dates[i])

        previous = repo_df[
            repo_df["snapshot_date"] == previous_date
        ][["package_path"] + metrics].copy()

        current = repo_df[
            repo_df["snapshot_date"] == current_date
        ][["package_path"] + metrics].copy()

        # Only compare packages present in both adjacent snapshots
        matched = previous.merge(
            current,
            on="package_path",
            how="inner",
            suffixes=("_previous", "_current"),
            validate="one_to_one",
        )

        if matched.empty:
            continue

        matched["repository"] = repository
        matched["previous_snapshot"] = previous_date
        matched["current_snapshot"] = current_date

        # Calculate the symmetric relative change for each metric
        changes = []

        for metric in metrics:
            old = matched[f"{metric}_previous"].astype(float)
            new = matched[f"{metric}_current"].astype(float)

            denominator = old.abs() + new.abs()

            relative_change = np.where(
                denominator == 0,
                0.0,
                (new - old).abs() / denominator,
            )

            matched[f"change_{metric}"] = relative_change
            changes.append(relative_change)

        matched["mean_structural_change"] = np.column_stack(
            changes
        ).mean(axis=1)

        matched["structural_change_proxy"] = (
            matched["mean_structural_change"] >= 0.30
        ).astype(int)

        # Predictors use only the previous snapshot
        for metric in metrics:
            matched[f"previous_{metric}"] = matched[
                f"{metric}_previous"
            ]

        transition_frames.append(matched)

transitions = pd.concat(transition_frames, ignore_index=True)

print("Matched package transitions:", len(transitions))
print("Repositories:", transitions["repository"].nunique())
print(
    "Positive proxy labels:",
    int(transitions["structural_change_proxy"].sum())
)
print(
    "Positive-label rate:",
    round(transitions["structural_change_proxy"].mean(), 4)
)

display(
    transitions.groupby("repository").agg(
        transitions=("package_path", "size"),
        positive_labels=("structural_change_proxy", "sum"),
        mean_change=("mean_structural_change", "mean"),
    ).reset_index()
)

Matched package transitions: 3379
Repositories: 11
Positive proxy labels: 275
Positive-label rate: 0.0814


,repository,transitions,positive_labels,mean_change
0,R01_spring_petclinic,7,4,0.295135
1,R02_common_lang,31,5,0.197248
2,R03_jabref,329,37,0.141140
3,R04_argouml,143,24,0.153910
4,R05_lucene,409,41,0.137072
5,R06_hadoop,1006,43,0.077223
6,R07_junit,106,16,0.161381
7,R08_apache_maven,123,18,0.167713
8,R09_kafca,328,39,0.130625
9,R10_teammates,22,6,0.271390


In [15]:
from pathlib import Path

output_dir = Path("/content/hrim_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "hrim_historical_transitions.csv"

transitions.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(transitions))
print("Columns:", len(transitions.columns))
print("File size (KB):", round(output_path.stat().st_size / 1024, 2))

Saved: /content/hrim_outputs/hrim_historical_transitions.csv
Rows: 3379
Columns: 30
File size (KB): 806.2


Train and evaluate the historical baseline

In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

# Load the saved historical transitions
df = pd.read_csv("/content/hrim_outputs/hrim_historical_transitions.csv")

# Use only previous-snapshot metrics as predictors
features = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

X = df[features].astype(float)
y = df["structural_change_proxy"].astype(int)
groups = df["repository"]

logo = LeaveOneGroupOut()
results = []
all_predictions = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_repo = groups.iloc[test_idx].iloc[0]

    # Handle the class imbalance using the training fold only
    positives = int(y_train.sum())
    negatives = len(y_train) - positives
    scale_weight = negatives / max(positives, 1)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    # ROC-AUC is undefined if the test set contains only one class
    roc_auc = (
        roc_auc_score(y_test, probabilities)
        if y_test.nunique() == 2
        else np.nan
    )

    row = {
        "repository": test_repo,
        "test_samples": len(test_idx),
        "positive_labels": int(y_test.sum()),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "pr_auc": average_precision_score(y_test, probabilities),
        "roc_auc": roc_auc,
    }

    results.append(row)

    all_predictions.append(pd.DataFrame({
        "repository": test_repo,
        "actual": y_test.to_numpy(),
        "probability": probabilities,
        "predicted": predictions,
    }))

results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

print("Per-repository results:")
display(results_df.round(4))

print("\nMean metrics across repositories:")
display(
    results_df[
        ["precision", "recall", "f1", "pr_auc", "roc_auc"]
    ].mean().to_frame("mean").round(4)
)

results_df.to_csv(
    "/content/hrim_outputs/hrim_loro_baseline_metrics.csv",
    index=False,
)

predictions_df.to_csv(
    "/content/hrim_outputs/hrim_loro_baseline_predictions.csv",
    index=False,
)

print("\nSaved metrics and predictions in /content/hrim_outputs/")

Per-repository results:


,repository,test_samples,positive_labels,precision,recall,f1,pr_auc,roc_auc
0,R01_spring_petclinic,7,4,0.6667,1.0000,0.8000,1.0000,1.0000
1,R02_common_lang,31,5,0.2083,1.0000,0.3448,0.4267,0.8231
2,R03_jabref,329,37,0.2453,0.3514,0.2889,0.2549,0.7058
3,R04_argouml,143,24,0.2105,0.3333,0.2581,0.1756,0.4958
4,R05_lucene,409,41,0.1682,0.4390,0.2432,0.2081,0.5997
5,R06_hadoop,1006,43,0.1034,0.3488,0.1596,0.1283,0.7148
6,R07_junit,106,16,0.2195,0.5625,0.3158,0.4171,0.6802
7,R08_apache_maven,123,18,0.1875,0.5000,0.2727,0.2750,0.6439
8,R09_kafca,328,39,0.1977,0.4359,0.2720,0.2164,0.6646
9,R10_teammates,22,6,0.3750,0.5000,0.4286,0.6267,0.7292



Mean metrics across repositories:


,mean
precision,0.2451
recall,0.5471
f1,0.3247
pr_auc,0.3471
roc_auc,0.7040



Saved metrics and predictions in /content/hrim_outputs/


Calculate the pooled evaluation metrics

In [17]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Use the out-of-repository predictions already created
pred = predictions_df.copy()

y_true = pred["actual"]
y_pred = pred["predicted"]
y_prob = pred["probability"]

print("Pooled evaluation metrics")
print("-------------------------")
print(f"Test samples: {len(pred)}")
print(f"Actual positive labels: {int(y_true.sum())}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"PR-AUC:    {average_precision_score(y_true, y_prob):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(y_true, y_pred, zero_division=0))

Pooled evaluation metrics
-------------------------
Test samples: 3379
Actual positive labels: 275
Precision: 0.1636
Recall:    0.4509
F1-score:  0.2401
PR-AUC:    0.1624
ROC-AUC:   0.6713

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[2470  634]
 [ 151  124]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.80      0.86      3104
           1       0.16      0.45      0.24       275

    accuracy                           0.77      3379
   macro avg       0.55      0.62      0.55      3379
weighted avg       0.88      0.77      0.81      3379



establish the baseline comparison

In [18]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

output_dir = Path("/content/hrim_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

y_true = predictions_df["actual"]
y_pred = predictions_df["predicted"]
y_prob = predictions_df["probability"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

report = {
    "model": "Historical six-metric XGBoost baseline",
    "validation": "Leave-one-repository-out",
    "target": "structural_change_proxy",
    "test_samples": int(len(y_true)),
    "positive_labels": int(y_true.sum()),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "pr_auc": float(average_precision_score(y_true, y_prob)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "features": [
        "previous_java_files",
        "previous_nonempty_lines",
        "previous_fan_in",
        "previous_fan_out",
        "previous_total_coupling",
        "previous_instability",
    ],
}

report_path = output_dir / "hrim_historical_baseline_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nSaved report to:", report_path)

{
  "model": "Historical six-metric XGBoost baseline",
  "validation": "Leave-one-repository-out",
  "target": "structural_change_proxy",
  "test_samples": 3379,
  "positive_labels": 275,
  "accuracy": 0.767682746374667,
  "precision": 0.16358839050131926,
  "recall": 0.4509090909090909,
  "f1": 0.24007744433688286,
  "pr_auc": 0.16237892128228193,
  "roc_auc": 0.6713097469540769,
  "true_negatives": 2470,
  "false_positives": 634,
  "false_negatives": 151,
  "true_positives": 124,
  "features": [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability"
  ]
}

Saved report to: /content/hrim_outputs/hrim_historical_baseline_report.json
